# 1 Load data

In [3]:
import pandas as pd
import datasets

dataset = datasets.load_dataset("McAuley-Lab/Amazon-Reviews-2023", "5core_timestamp_Books")
metadata = datasets.load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Books")

# load 50% of the data
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])
meta_df = pd.DataFrame(metadata["full"])

# drop duplicates 
meta_df = meta_df.drop_duplicates(subset=['parent_asin'])
metadata_filtered_df = meta_df[meta_df.parent_asin.isin(train_df.parent_asin)]


# 2 Process data

In [ ]:
import re 
def create_source_text(product):
        """Concatenate product information into a text string

        Args:
            product (dict): dictionary containing product information

        Returns:
            str: concatenated product information
        """
        description = "*" if product["description"] == "" else f"Description: {product['description']}"
        features = "*" if product["features"] == "" else f"Features: {product['features']}"
        details = "*" if product["details"] == "" else f"Details: {product['details']}"
        store = "*" if product["store"] == "" else f"Store: {product['store']}"
        categories = "*" if product["categories"] == "" else f"Categories: {product['categories']}"
        price = "*" if product["price"] == "" else f"Price: {product['price']}"
        author = "*" if product["author"] == "" else f"Author: {product['author']}"

        # concatenated_text = f"Parent ASIN: {product['parent_asin']}; Title: {product['title']}; Author: {author}; Description: {description}; Features: {features} - {details}; Store: {store}; Categories: {categories}; Price: {price};"
        concatenated_text = f"Parent ASIN: {product['parent_asin']}; Title: {product['title']}Description: {description};- {details}; Store: {store};"
        #print("Concatenated text:", concatenated_text)  # Debug: Print the output text
        return concatenated_text
    
def clean_string(string):
            string = re.sub(r'\[', '', string)
            string = re.sub(r'\]', '', string)
            string = re.sub(r'"', '', string)
            string = re.sub(r'\s+', ' ', string)
            string = re.sub("{", "", string)
            string = re.sub("}", "", string)
            return string

def clean_column(df, column_name):
    """
    Clean a column in the dataframe by applying clean_string to each element.
    Handles lists by converting them to strings first.
    
    Args:
        df (pandas.DataFrame): The dataframe containing the column to clean
        column_name (str): The name of the column to clean
        
    Returns:
        pandas.Series: The cleaned column
    """
    def safe_clean(value):
        if isinstance(value, list):
            # Convert list to string before cleaning
            return clean_string(str(value))
        elif pd.isna(value) or value is None:
            return ""
        else:
            return clean_string(str(value))
    
    return df[column_name].apply(safe_clean)

# Clean text columns that might contain lists
for column in ['details', 'features', 'categories', 'description', 'bought_together']:
    if column in metadata_filtered_df.columns:
        metadata_filtered_df[column] = clean_column(metadata_filtered_df, column)

metadata_filtered_df['source_text'] = metadata_filtered_df.apply(create_source_text, axis=1)


# 3 Hierarchies approaches
## 3.1 KeyBert / COL LLM approach
+ Uses key word extraction, llms to build hierarchies. 
+ Task NER, Relation extraction, Relation classification

KeyBert 

Sideote:
- KeyBert extract keywords 
    + use sentence transformer for creating candidates 
    + use KeyBert for keyword extracting entities

- Used sentence_transformer: 
    + KeyBert use as default All-MiniLM-L6-v2
    + [1]: https://ieeexplore.ieee.org/document/10295108 shows that using paraphrase-mpnet-base-v2 shows good results, model is trained by microsoft for keyword extraction 
        + keyword extraction compared to key phrase extraction better results (f1, rec, pre, map)
        + peak f1 score of 0.82 at 50 token length, decline after 50 token length and crash if document has 1000 tokens 
 

Explore if paraphrase-mpnet-base-v2 extracts more informative keywords for hierarchies than all-MiniLM-L6-v2 

In [ ]:
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer

sentence_transformer = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")
kw_model = KeyBERT(model=sentence_transformer)

test = kw_model.extract_keywords(metadata_filtered_df.iloc[1].source_text, keyphrase_ngram_range=(1, 1), top_n=10,stop_words="english")

test

In [ ]:
sentence_transformer = SentenceTransformer("all-MiniLM-L6-v2")
kw_model = KeyBERT(model=sentence_transformer)

test = kw_model.extract_keywords(metadata_filtered_df.iloc[1].source_text, keyphrase_ngram_range=(1, 1), top_n=10,stop_words="english")

test

We can see that paraphrase-mpnet-base-v2 extracts more informative keywords for hierarchies than all-MiniLM-L6-v2 

+ less noisy key words, all-MiniLM-L6-v2 extracts more key words with small information (e. g. "oz", "ounces")



Explore if n-gramm range has an impact on the quality of the keywords 

Assumption:
+ higher n-gramm range, more specific keywords 
+ lower n-gramm range, more general keywords 



In [ ]:
sentence_transformer = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")
kw_model = KeyBERT(model=sentence_transformer)

test = kw_model.extract_keywords(metadata_filtered_df.iloc[1].source_text, keyphrase_ngram_range=(1, 2), top_n=10,stop_words="english")

test

We can see that a n-gramm range of (1,2) seems to extract very similar keywords. For instance spray and mist from the (1,1) range are also combined in the (1,2) range as sprayer mist. But this keyword is very similar to spray, glass spray. Therefore the informative gain of the key words decreases with higher n-gramm range.

Therefore we explore if a maxssum parameter has an impact on the quality of the keywords 

In [6]:
from langchain_core.prompts import ChatPromptTemplate 
from langchain_ollama import OllamaLLM

LLM_MODEL = "llama3.2"

llm= OllamaLLM(model=LLM_MODEL,temperature=0.1)

In [ ]:
detailed_summary_prompt = """You are an expert that generates detailed product descriptions. 
You receive a short compact product description. The detailed product description will be later used to extract Named Entities or Key Words to build a taxonomy

Side information: the product description includes ingredients (e.g almond oil), usage information (e. g. silky hair), packe dimensions etc
Be creative. Make it about a paragraph long

Rules:
- Dont include explainations or notes
- Dont hallucinate 
- Only add valuable information to the product
- Priorities sizing or packaging dimensions less, but still include them
"""

detail_user_prompt = """
This is the description of the product. {product_description}
"""

detail_prompt = ChatPromptTemplate(
    messages=[("system", detailed_summary_prompt),
              ("human",detail_user_prompt)], 
              input_variables = ["product_description"]

)

detail_chain = detail_prompt | llm 

detailed_summary = detail_chain.invoke({"product_description":metadata_filtered_df.iloc[0].source_text})
detailed_summary

In [ ]:
sentence_transformer = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")
kw_model = KeyBERT(model=sentence_transformer)
raw_keywords = kw_model.extract_keywords(metadata_filtered_df.iloc[0].source_text, keyphrase_ngram_range=(1, 2), top_n=10,stop_words="english", use_maxsum=True)

filtered_keywords = []
for i in range(0,len(raw_keywords)):
    kw = raw_keywords[i][0]
    filtered_keywords.append(kw)
    filtered_keywords = list(set(filtered_keywords))

filtered_keywords

In [ ]:
raw_keywords = kw_model.extract_keywords(detailed_summary, keyphrase_ngram_range=(1, 2), top_n=10,stop_words="english", use_maxsum=True)

filtered_keywords = []
for i in range(0,len(raw_keywords)):
    kw = raw_keywords[i][0]
    filtered_keywords.append(kw)
    filtered_keywords = list(set(filtered_keywords))

filtered_keywords

In [10]:
ROOT_ENTITY = "Product details"
ROOT_ENTITY = metadata_filtered_df.iloc[0].title

Playground for Chain of Layers LLM approach


+ 1 Summarization 
+ 2 Generation of entities (llm o. ner model)
+ 3 Generation of tax
+ 4 update of taxonomy 
+ 5 review 


In [ ]:
gen_tax_sys_prompt = """
You are an helpful assistant and expert in constructing a taxonomy from a given concept.
Build a taxonomy where the root concept is a {root_entity}

The format of the generated taxonomy is: 1. Parent Concept 1.1 Child Concept.
Do not change any entity names when building the taxonomy.

Critical rules 
- Do not add any comments or explanations 
- There is one and only one root node of the taxonomy 
- All entities from the list must apper in the taxonomy if feasible
- make the nodes as specific as possible
- Disclude entities with no information detail for the Parent Concept
- you allowed to add other entities if they make sense, dont hallucinate or overfit the taxonomy

Few examples 
Demonstration #1 
Input: ['organic sweet',
 'oil bundle',
 'package dimensions',
 'fractionated coconut',
 'free moisturizing',
 'silky hair',
 'shiny leaf',
 'hair',
 'items package',
 'almond']
Output: 
1. organic sweet oil bundle
1.1 usage 
1.1.1 hair 
1.1.2 silky hair
1.2 ingredients 
1.2.1 almond 
1.2.2 fractionated coconut
1.3 features
1.3.1 oil bundle 
"""
gen_tax_input_message = """
The root entity is {root_entity} and entitiy list contains following words {list_entities}
The format of the generated taxonomy should be: 1. Parent Concept 1.1 Child Concept.
Dont add any comments or explanations.

"""

gen_prompt = ChatPromptTemplate(messages=[
    ("system", gen_tax_sys_prompt),
    ("user", gen_tax_input_message)
    ], 
            input_variables=["root_entity", "list_entities"]
        )

gen_chain =  gen_prompt | llm 

gen_taxonomy = gen_chain.invoke({"root_entity": ROOT_ENTITY, "list_entities":filtered_keywords})

gen_taxonomy

In [ ]:
update_system_prompt = """
You are an helpful assistant and expert in updating a hierarchical taxonomy from a given concept.
Review a given hierarchical taxnonomy which got constructed by an llm from a list of entities and a given root concept

If the taxnonomy contains unreasonable or wrong relations, update those to make sense. 
Also you are allowed to relocate child nodes to new parents if it makes sense.
If neccessary add parent or child nodes.

Critical rules 
- DO NOT add any comments or explanations 
- There is one and only one root node of the taxonomy 
- All entities from the list must apper in the taxonomy if feasible
- Make the nodes as specific as possible
- Disclude entities with no information detail for the Parent Concept
- Keep the numbering format

"""

update_user_prompt = """
The root entity is {root_concept} and entitiy list contains following words {list_entities}. 
The last LLM generated following taxonomy: {taxonomy}

"""

update_prompt = ChatPromptTemplate(

    messages=[("system",update_system_prompt),
              ("user"),update_user_prompt],
    input_variables=["root_concept","list_entities","taxonomy"]
)

update_chain = update_prompt | llm 

updated_taxonomy = update_chain.invoke({"root_concept":ROOT_ENTITY,"list_entities":filtered_keywords, "taxonomy":gen_taxonomy})

updated_taxonomy

In [ ]:
review_sys_prompt = """
You are an expert for validating a generated hierarchical taxonomy. 
The taxonomy was generated from a given parent concept, a list of key entities by an LLM. 

Your task is to classify the given hierarchical taxonomy. Look at the product description and taxonomy. 

If the taxnonomy is not suitable for the product, for example because it  contains unreasonable or wrong relations, update those to make sense. 
Return a FALSE as output.
If the taxonomy is fine return a TRUE as output. 


Critical rules: 
- DO NOT add any comments or explanations 
- ONLY RETURN THE BOOLEAN 
- Do not hallucinate 
- Validate carefully the taxonomy.

Output Format: 
{{BOOLEAN}}
"""

review_user_prompt = """
The root entity is {root_concept} and entitiy list contains following words {list_entities}.  
The last LLM returned this updated taxonomy: {taxonomy}
Product description: {product_description}
"""

review_prompt = ChatPromptTemplate(

    messages=[("system",review_sys_prompt),
              ("user"),review_user_prompt],
    input_variables=["root_concept","list_entities","taxonomy", "product_description"]
)

review_chain = review_prompt | llm

reviewed_taxonomy_bool = review_chain.invoke({"root_concept":ROOT_ENTITY,"list_entities":filtered_keywords, "taxonomy":updated_taxonomy,"product_description":detailed_summary})

reviewed_taxonomy_bool


In [ ]:
updated_taxonomy

In [ ]:
# post process taxonomy into triples format 
def taxonomy_to_triples(taxonomy_text):
    """
    Convert a hierarchical taxonomy text into a list of triples (head, relation, tail).
    
    Args:
        taxonomy_text (str): The hierarchical taxonomy in text format like:
        '2. Product details\n 2.1 Skincare\n   2.1.1 Effective skincare\n...'
        
    Returns:
        DataFrame: DataFrame with 'head', 'relation', and 'tail' columns
    """
    lines = taxonomy_text.strip().split('\n')
    triples = []
    
    # Track the hierarchy levels and their corresponding concepts
    hierarchy = {}
    
    for line in lines:
        # Skip empty lines
        if not line.strip():
            continue
        
        # Extract the level number and concept
        parts = line.strip().split(' ', 1)
        if len(parts) < 2:
            continue
            
        level_num, concept = parts
        
        # Remove the trailing period from level_num if it exists
        level_num = level_num.rstrip('.')
        
        # Store the concept at its level
        hierarchy[level_num] = concept
        
        # If not the root level, create a parent-child relationship
        if '.' in level_num:
            # Get the parent level by removing the last part after the dot
            parent_level = '.'.join(level_num.split('.')[:-1])
            if parent_level in hierarchy:
                parent = hierarchy[parent_level]
                triples.append({
                    'head': parent,
                    'relation': 'is parent of',
                    'tail': concept
                })
                
    import pandas as pd
    return pd.DataFrame(triples, columns=['head', 'relation', 'tail'])

# Process the updated_taxonomy
taxonomy_triples = taxonomy_to_triples(updated_taxonomy)
taxonomy_triples


### Explore Prompt Compression 

Sidenote: 
- Prompt compression is a technique to reduce the size of a prompt by removing redundant information or simplifying the language. 
- This can help reduce the cost of generating responses, as well as improve the performance of the model by reducing the amount of data it needs to process.
- Prompt compression can be achieved through various methods, such as removing unnecessary tokens, using more concise language, or using pre-defined templates.


In [ ]:
from llmlingua import PromptCompressor


llm_lingua = PromptCompressor(
    use_llmlingua2=False,  # Whether to use llmlingua-2
    device_map="cpu"  # Force CPU usage instead of CUDA
)
gen_tax_sys_prompt_compressed = llm_lingua.compress_prompt(gen_tax_sys_prompt, rate=0.33, force_tokens = ['\n', '?'])["compressed_prompt"]
update_sys_prompt_compressed = llm_lingua.compress_prompt(update_system_prompt, rate=0.33, force_tokens = ['\n', '?'])["compressed_prompt"] 
review_sys_prompt_compressed = llm_lingua.compress_prompt(review_sys_prompt, rate=0.33, force_tokens = ['\n', '?'])["compressed_prompt"]


import Levenshtein 

ratio_gen_tax_sys_prompt = Levenshtein.ratio(gen_tax_sys_prompt_compressed, gen_tax_sys_prompt)
ratio_update_sys_prompt = Levenshtein.ratio(update_sys_prompt_compressed, update_system_prompt)
ratio_review_sys_prompt = Levenshtein.ratio(review_sys_prompt_compressed, review_sys_prompt)

print("Similarity of Generate Taxonomy System Prompt and Compressed System Prompt: ", ratio_gen_tax_sys_prompt)
print("Similarity of Update System Prompt and Compressed System Prompt: ", ratio_update_sys_prompt)
print("Similarity of Review System Prompt and Compressed System Prompt: ", ratio_review_sys_prompt)

In [ ]:
import Levenshtein 

print("Similarity of Generate Taxonomy System Prompt and Compressed System Prompt: ", Levenshtein.ratio(gen_tax_sys_prompt_compressed, gen_tax_sys_prompt))
print("Similarity of Update System Prompt and Compressed System Prompt: ", Levenshtein.ratio(update_sys_prompt_compressed, update_system_prompt))
print("Similarity of Review System Prompt and Compressed System Prompt: ", Levenshtein.ratio(review_sys_prompt_compressed, review_sys_prompt))


Use LLMLingua2, which uses smaller Language Models to compress the prompt. 

In [26]:
from llmlingua import PromptCompressor

llm_lingua = PromptCompressor(
    model_name="microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank",
    use_llmlingua2=True,  # Whether to use llmlingua-2
    device_map="cpu"  # Force CPU usage instead of CUDA
)
gen_sys_prompt_compressed = llm_lingua.compress_prompt(gen_tax_sys_prompt, rate=0.33, force_tokens = ['\n', '?'])["compressed_prompt"]
update_sys_prompt_compressed = llm_lingua.compress_prompt(update_system_prompt, rate=0.33, force_tokens = ['\n', '?'])["compressed_prompt"] 
review_sys_prompt_compressed = llm_lingua.compress_prompt(review_sys_prompt, rate=0.33, force_tokens = ['\n', '?'])["compressed_prompt"]


Similarity of Taxonomy System Prompt and Compressed System Prompt

In [ ]:
import Levenshtein 

print("Similarity of Generate Taxonomy System Prompt and Compressed System Prompt: ", Levenshtein.ratio(gen_tax_sys_prompt_compressed, gen_tax_sys_prompt))
print("Similarity of Update System Prompt and Compressed System Prompt: ", Levenshtein.ratio(update_sys_prompt_compressed, update_system_prompt))
print("Similarity of Review System Prompt and Compressed System Prompt: ", Levenshtein.ratio(review_sys_prompt_compressed, review_sys_prompt))


In [ ]:
gen_tax_sys_prompt


In [ ]:
gen_tax_sys_prompt_compressed

In [ ]:
review_sys_prompt.split("\n")

In [ ]:
review_sys_prompt_compressed.split("\n")

In [ ]:
# generate a new taxonomy 

gen_tax_prompt_compressed = gen_tax_sys_prompt_compressed + gen_tax_input_message


gen_compressed_chain = ChatPromptTemplate(
    messages=[("system", gen_sys_prompt_compressed),
              ("user", gen_tax_input_message)],
    input_variables=["root_entity", "list_entities"]
) | llm 

gen_compressed_taxonomy = gen_compressed_chain.invoke({"root_entity":ROOT_ENTITY,"list_entities":filtered_keywords})

gen_compressed_taxonomy


Tokens saved 

Sidenote: 
Llama 3.2 uses BPE tokenizer. We would need to load the model directly to use the tokenizer. Because this method is more computationally expensive, we use the cl100k_base tokenizer (also used by GPT-3 - GPT-o1 w/o GPT-4o). This gives us a fast and good approximation of the number of tokens. 

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")


tokens_regular = len(encoding.encode(gen_tax_sys_prompt))
tokens_compressed = len(encoding.encode(gen_sys_prompt_compressed))

print("Tokens regular: ", tokens_regular)
print("Tokens compressed: ", tokens_compressed)
print("Tokens saved: ", tokens_regular - tokens_compressed)
print("Tokens saved : ", (tokens_regular / tokens_compressed) * 100,"%")

**Interpreation**: _It is noticable that the compressed prompt flattens the hierarchy to a single level with one parent and ten childrens. This granularity of taxonomy is of low information depth. Additionally, the children are very specfic. If we use the compressed prompts for all products, the taxonomy will be very specific and not generalizable. Therefore, we will get a taxonomy tree with little connections between nodes from different products._

**Implications**: For further experiments we will not use the compressed prompts for the taxonomy generation. 


### Build LLM Class 

In [44]:
gen_tax_sys_prompt_compressed = llm_lingua.compress_prompt(gen_tax_sys_prompt, rate=0.33, force_tokens = ['\n', '?'])
update_sys_prompt_compressed = llm_lingua.compress_prompt(update_system_prompt, rate=0.33, force_tokens = ['\n', '?'])
review_sys_prompt_compressed = llm_lingua.compress_prompt(review_sys_prompt, rate=0.33, force_tokens = ['\n', '?'])


In [1]:
from langchain_core.prompts import ChatPromptTemplate 
from langchain_ollama import OllamaLLM
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
import pandas as pd
import os
import time
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class LLMHierarchyCOL: 

    def __init__(self,llm_name) -> None:
        self.llm = OllamaLLM(model=llm_name, temperature=0.1)

    def generate_details(self,short_summary:str)->str:
        """
        Generate a detailed product description from a short summary
        Args:
            short_summary (str): The short summary of the product
            detailed_summary_prompt (str): The prompt for the detailed summary (standard prompt or compressed prompt)
        Returns:
            str: The detailed product description
        """
        
        detailed_summary_prompt = """You are an expert that generates detailed product descriptions. 
        You receive a short compact product description. The detailed product description will be later used to extract Named Entities or Key Words to build a taxonomy

        Side information: the product description includes ingredients (e.g almond oil), usage information (e. g. silky hair), packe dimensions etc
        Be creative. Make it about a paragraph long

        Rules:
        - Dont include explainations or notes
        - Dont hallucinate 
        - Only add valuable information to the product
        - Priorities sizing or packaging dimensions less, but still include them
        """
        detail_user_prompt = """
        This is the description of the product. {product_description}
        """

        detail_prompt = ChatPromptTemplate(
            messages=[("system", detailed_summary_prompt),
                    ("human",detail_user_prompt)], 
                    input_variables = ["product_description"]

        )

        detail_chain = detail_prompt | self.llm 

        detailed_summary = detail_chain.invoke({"product_description":short_summary})
        return detailed_summary

    def get_keywords(self,sentence_transformer:str, detailed_summary:str)-> list:
        sentence_transformer = SentenceTransformer(sentence_transformer)
        kw_model = KeyBERT(model=sentence_transformer)
        raw_keywords = kw_model.extract_keywords(detailed_summary, keyphrase_ngram_range=(1, 2), top_n=10,stop_words="english", use_maxsum=True)

        filtered_keywords = []
        for i in range(0,len(raw_keywords)):
            kw = raw_keywords[i][0]
            filtered_keywords.append(kw)
            filtered_keywords = list(set(filtered_keywords))

        filtered_keywords  
        return filtered_keywords

    def generate_tax(self, root_concept:str, list_keywords: list, taxonomy:str)->str:
        """
        Generate a taxonomy from a given concept and list of keywords
        Args:
            root_concept (str): The root concept of the taxonomy
            list_keywords (list): The list of keywords to be included in the taxonomy
            generate_tax_system_prompt (str): The prompt for the taxonomy generation (standard prompt or compressed prompt)
            taxonomy (str): The taxonomy generated in the previous step
        Returns:
            str: The generated taxonomy
        """
        generate_system_prompt = """
        You are an helpful assistant and expert in constructing a taxonomy from a given concept.
        You will build iteratively a taxonomy with a root concept and from the taxonomy in the previous step and the entity list

        The format of the generated taxonomy is: 1. Parent Concept 1.1 Child Concept.
        Do not change any entity names when building the taxonomy.

        Critical rules 
        - DO NOT ADD ANY COMMENTS OR EXPLANATIONS 
        - THERE IS ONE AND ONLY ONE ROOT NODE OF THE TAXONOMY 
        - ALL ENTITIES FROM THE LIST MUST APPER IN THE TAXONOMY IF FEASIBLE
        - DISCLUDE ENTITIES WITH NO INFORMATION DETAIL FOR THE PARENT CONCEPT
        - YOU ALLOWED TO ADD OTHER ENTITIES IF THEY MAKE SENSE, DONT HALLUCINATE OR OVERFIT THE TAXONOMY
        - ONLY RETURN THE TAXONOMY

        Few examples 
        Demonstration #1 
        Input: ['organic sweet',
        'oil bundle',
        'package dimensions',
        'fractionated coconut',
        'free moisturizing',
        'silky hair',
        'shiny leaf',
        'hair',
        'items package',
        'almond']
        Output: 
        1. organic sweet oil bundle
        1.1 usage 
        1.1.1 hair 
        1.1.2 silky hair
        1.2 ingredients 
        1.2.1 almond 
        1.2.2 fractionated coconut
        1.3 features
        1.3.1 oil bundle 
        """
        gen_tax_input_message = """
        The root entity is {root_concept}, the taxonomy in the current step is {taxonomy} and entitiy list of the current step contains following words {list_entities}
        """

        gen_prompt = ChatPromptTemplate(messages=[
            ("system", generate_system_prompt),
            ("user", gen_tax_input_message)
            ], 
                    input_variables=["root_entity", "list_entities","taxonomy"]
                )

        gen_chain =  gen_prompt | self.llm 

        gen_taxonomy = gen_chain.invoke({"root_concept": root_concept, "list_entities":list_keywords,"taxonomy":taxonomy})

        gen_taxonomy
        return gen_taxonomy
    
    def update_tax(self, taxonomy:str,root_concept:str, list_keywords: list)->str:
        """
        Update a taxonomy from a given concept and list of keywords
        Args:
            taxonomy (str): The taxonomy generated in the previous step
            root_concept (str): The root concept of the taxonomy
            list_keywords (list): The list of keywords to be included in the taxonomy
            update_system_prompt (str): The prompt for the taxonomy update (standard prompt or compressed prompt)
        Returns:
            str: The updated taxonomy
        """
        update_system_prompt = """
            You are an helpful assistant and expert in updating a given hierarchical taxonomy with a root concept.
            Review a given hierarchical taxnonomy from the previous step which got constructed by an llm from a list of entities and a given root concept

            If the taxnonomy contains unreasonable or wrong relations, update those to make sense. 
            Also you are allowed to relocate child nodes to new parents if it makes sense.
            If neccessary add parent or child nodes.

            Critical rules 
            - DO NOT ADD ANY COMMENTS OR EXPLANATIONS 
            - THERE IS ONE AND ONLY ONE ROOT NODE OF THE TAXONOMY 
            - ALL ENTITIES FROM THE LIST MUST APPER IN THE TAXONOMY IF FEASIBLE
            - DISCLUDE ENTITIES WITH NO INFORMATION DETAIL FOR THE PARENT CONCEPT
            - KEEP THE NUMBERING FORMAT
            - ONLY RETURN THE TAXONOMY

            """

        update_user_prompt = """
        The root entity is {root_concept} and entitiy list contains following words {list_entities}. 
        The last LLM generated following taxonomy: {taxonomy}

        """

        update_prompt = ChatPromptTemplate(

            messages=[("system",update_system_prompt),
                    ("user"),update_user_prompt],
            input_variables=["root_concept","list_entities","taxonomy"]
        )

        update_chain = update_prompt | self.llm 

        updated_taxonomy = update_chain.invoke({"root_concept":root_concept,"list_entities":list_keywords, "taxonomy":taxonomy})

        updated_taxonomy
        return updated_taxonomy
    
    def review(self,detailed_summary,taxonomy:str,root_concept:str, list_keywords: list)->str:
        """
        Review a taxonomy from a given concept and list of keywords
        Args:
            detailed_summary (str): The detailed product description
            taxonomy (str): The taxonomy to be reviewed
            root_concept (str): The root concept of the taxonomy
            list_keywords (list): The list of keywords to be included in the taxonomy
            review_system_prompt (str): The prompt for the taxonomy review (standard prompt or compressed prompt)
        Returns:
            str: The reviewed taxonomy
        """
        review_system_prompt = """
        You are an expert for validating a generated hierarchical taxonomy. 
        The taxonomy was generated from a given parent concept, a list of key entities by an LLM. 

        Your task is to classify the given hierarchical taxonomy. Look at the product description and taxonomy. 

        If the taxnonomy is not suitable for the product, for example because it  contains unreasonable or wrong relations, update those to make sense. 
        Return a FALSE as output.
        If the taxonomy is fine return a TRUE as output. 


        Critical rules: 
        - Do not include any explanations or notes 
        - ONLY RETURN THE BOOLEAN 
        - Do not hallucinate 
        - Validate carefully the taxonomy.

        Output Format: 
        {{BOOLEAN}}
        """

        review_user_prompt = """
        The root entity is {root_concept} and entitiy list contains following words {list_entities}.  
        The last LLM returned this updated taxonomy: {taxonomy}
        Product description: {product_description}
        """

        review_prompt = ChatPromptTemplate(

            messages=[("system",review_system_prompt),
                    ("user"),review_user_prompt],
            input_variables=["root_concept","list_entities","taxonomy", "product_description"]
        )

        review_chain = review_prompt | self.llm

        reviewed_taxonomy_bool = review_chain.invoke({"root_concept":root_concept,"list_entities":list_keywords, "taxonomy":taxonomy,"product_description":detailed_summary})

        reviewed_taxonomy_bool
        return reviewed_taxonomy_bool 
    
        # post process taxonomy into triples format 
    def taxonomy_to_triples(self,taxonomy_text):
        """
        Convert a hierarchical taxonomy text into a list of triples (head, relation, tail).
        
        Args:
            taxonomy_text (str): The hierarchical taxonomy in text format like:
            '2. Product details\n 2.1 Skincare\n   2.1.1 Effective skincare\n...'
            
        Returns:
            DataFrame: DataFrame with 'head', 'relation', and 'tail' columns
        """
        lines = taxonomy_text.strip().split('\n')
        triples = []
        
        # Track the hierarchy levels and their corresponding concepts
        hierarchy = {}
        
        for line in lines:
            # Skip empty lines
            if not line.strip():
                continue
            
            # Extract the level number and concept
            parts = line.strip().split(' ', 1)
            if len(parts) < 2:
                continue
                
            level_num, concept = parts
            
            # Remove the trailing period from level_num if it exists
            level_num = level_num.rstrip('.')
            
            # Store the concept at its level
            hierarchy[level_num] = concept
            
            # If not the root level, create a parent-child relationship
            if '.' in level_num:
                # Get the parent level by removing the last part after the dot
                parent_level = '.'.join(level_num.split('.')[:-1])
                if parent_level in hierarchy:
                    parent = hierarchy[parent_level]
                    triples.append({
                        'head': parent,
                        'relation': 'is parent of',
                        'tail': concept
                    })
                    
        return pd.DataFrame(triples, columns=['head', 'relation', 'tail'])

    def linkage_asin_to_taxonomy(self,taxonomy_triples,dict_asin_keywords):
        """
        Link an asin to the taxonomy entities which have corresponding keywords
        Args:
        taxonomy_triples (DataFrame): The taxonomy in triples format
        metadata_filtered_df (DataFrame): The metadata filtered df
    Returns:
        DataFrame: DataFrame with 'parent_asin', 'taxonomy_entities' columns
        """


        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        head_embeddings = []
        tail_embeddings = []
        for _, row in taxonomy_triples.iterrows():
            head_embeddings.append(embedding_model.encode(row["head"]))
            tail_embeddings.append(embedding_model.encode(row["tail"]))
        taxonomy_triples["embedding_head"] = head_embeddings
        taxonomy_triples["embedding_tail"] = tail_embeddings

        new_triples = [] # list to store linked parent asin to taxonomy entities
        # loop over dict_asin_keywords
        for parent_asin, keywords in dict_asin_keywords.items():
            for keyword in keywords:
                print("Keyword:", keyword)
                kw_embedding = embedding_model.encode(keyword).reshape(1, -1)
    
            for idx, row in taxonomy_triples.iterrows():
                # Safely get embeddings (skip if invalid)
                try:
                    head_embed = np.array(row["embedding_head"]).reshape(1, -1)
                    tail_embed = np.array(row["embedding_tail"]).reshape(1, -1)
                    parent_asin = parent_asin
                except (AttributeError, ValueError):
                    continue
            
            # Calculate similarities
                head_sim = cosine_similarity(kw_embedding, head_embed)[0][0]
                tail_sim = cosine_similarity(kw_embedding, tail_embed)[0][0]
            
            # Add new triples if similarity > threshold
                if head_sim > 0.5:
                    new_triples.append({
                        "head": parent_asin,
                        "relation": "related to",
                        "tail": row["head"]
                    })
                elif tail_sim > 0.5:
                    new_triples.append({
                        "head": parent_asin,
                        "relation": "related to",
                        "tail": row["tail"]
                    })

        # Add all new triples at once (more efficient than concat in loop)
        if new_triples:
            taxonomy_triples = pd.concat([taxonomy_triples, pd.DataFrame(new_triples)], ignore_index=True)
            taxonomy_triples = taxonomy_triples.drop(columns=['embedding_head', 'embedding_tail'])
            taxonomy_triples = taxonomy_triples.drop_duplicates()
            taxonomy_triples

        return taxonomy_triples

if __name__ == "__main__":
    # Hyperparameters
    DATASETS = ["Books", "All_Beauty", "Beauty_and_Personal_Care"]
    DATASET = DATASETS[1]
    DIR_NAME = "amazon-" # else: Last_fm, MovieLens
    LLM_MODEL = "llama3.2"
    SENTENCE_TRANSFORMER = "sentence-transformers/paraphrase-mpnet-base-v2"
    ROOT_CONCEPT = "Product details"
    
    # Get the project root directory (Masterarbeit-Playground folder)
    current_dir = os.getcwd()
    
    #data_path = os.path.join(current_dir, 'data', 'preprocessed', f'{DIR_NAME}{DATASET}', 'metadata_filtered_df.csv')
    #data_path = os.path.join(current_dir, 'data', 'preprocessed', f'amazon-All_Beauty', 'metadata_filtered_df.csv')
    metadata_filtered_df = pd.read_csv("/Users/U725801/Documents/GitHub/Masterarbeit-Playground/data/preprocessed/amazon-All_Beauty/metadata_filtered_df.csv")

    start_time = time.time()
    # Initilaize the class
    llm_hierarchy_col = LLMHierarchyCOL(LLM_MODEL)
    
    # enhance summary with details
    for index, row in metadata_filtered_df.iterrows():
        short_summary = row.source_text
        detailed_summary = llm_hierarchy_col.generate_details(short_summary)
        metadata_filtered_df.at[index, 'detailed_summary'] = detailed_summary

    # generate taxonomy stepwise by using batch of 20 items 
    START_INDEX = 0
    BATCH_SIZE = 100
    list_keywords = []
    generated_taxonomy = ""
    dict_asin_keywords = {}
    for i in range(START_INDEX,len(metadata_filtered_df),BATCH_SIZE):
        batch_keywords = []
        for index in range(i,i+BATCH_SIZE):
            keywords = llm_hierarchy_col.get_keywords(SENTENCE_TRANSFORMER,metadata_filtered_df.iloc[index].detailed_summary)
            batch_keywords.append(keywords)
            list_keywords.append(keywords)
            dict_asin_keywords[metadata_filtered_df.iloc[index].parent_asin] = keywords
        print("batch_keywords",batch_keywords)

        # generate taxonomy
        generated_taxonomy = llm_hierarchy_col.generate_tax(ROOT_CONCEPT,batch_keywords,generated_taxonomy)
        print("gen_tax",generated_taxonomy)
        update_tax = llm_hierarchy_col.update_tax(generated_taxonomy,ROOT_CONCEPT, batch_keywords)
        print("update_tax",update_tax)
        review_status = llm_hierarchy_col.review(metadata_filtered_df.iloc[i].detailed_summary,update_tax,ROOT_CONCEPT,batch_keywords)
        print("review_status",review_status)
        if review_status == False:
            print("Taxonomy validation failed. Regenerating taxonomy...")
            updated_tax = llm_hierarchy_col.generate_tax(ROOT_CONCEPT, batch_keywords)
            review_status = llm_hierarchy_col.review(metadata_filtered_df.iloc[i].detailed_summary,updated_tax,ROOT_CONCEPT,batch_keywords)
            print("updated_tax",updated_tax)
            print("review_status",review_status)
        else:
            print("Taxonomy validation successful")

        # post process taxonomy into triples format 
        taxonomy_triples_df = llm_hierarchy_col.taxonomy_to_triples(update_tax)
        taxonomy_triples_df = llm_hierarchy_col.linkage_asin_to_taxonomy(taxonomy_triples_df,dict_asin_keywords)

    # save taxonomy_triples
    taxonomy_triples_df.to_csv(os.path.join("/Users/U725801/Documents/GitHub/Masterarbeit-Playground", 'data', 'relations', f'{DIR_NAME}{DATASET}', f'{LLM_MODEL}_taxonomy_triples.csv'), index=False)
    end_time = time.time()
    print("Runtime: ", end_time - start_time)
    print("Runtime per item in seconds: ", (end_time - start_time) / len(metadata_filtered_df))
    


batch_keywords [['luxurious blend', 'effective moisturizer', 'organic sweet', 'blend nourishing', 'leaf luxurious', 'fractionated coconut', 'oil used', 'almond', 'shiny leaf', 'skin hair'], ['perfect refilling', 'sprayers feature', 'refilling', 'pack cherry', 'spray', 'bottles2 pack', 'convenient mist', 'glass spray', 'preferred liquid', 'bottles eco'], ['locks versatile', 'size shower', 'comb provides', 'hair types', 'hair silky', 'shower', 'straight locks', 'comb set', 'tooth shower', 'bathroom cabinet'], ['lasting hydration', 'smooth skin', 'probiotic skin', 'skin natural', 'tula probiotic', 'cream luxurious', '24 hydration', 'cream fits', 'hydration anti', 'night cream'], ['fine hair', 'ikoco pack', 'stylish hair', 'jaw clips', 'clamps expertly', 'clips come', 'hairstyle making', 'pack jaw', 'hold hair', 'clips provide'], ['facial cleanser', 'thoughtful gift', 'facial makeup', 'box facial', 'kit women', 'care package', 'kit includes', 'golden gift', 'women', 'effective skincare'], 

RuntimeError: MPS backend out of memory (MPS allocated: 18.14 GB, other allocations: 16.02 MB, max allowed: 18.13 GB). Tried to allocate 3.00 KB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## 3.2 Hierarchy Embeddings

In [ ]:
from hierarchy_transformers import HierarchyTransformer

model = HierarchyTransformer.from_pretrained('Hierarchy-Transformers/HiT-MiniLM-L12-WordNetNoun')
entity_embeddings = model.encode(lst_keywords)

entity_embeddings

In [ ]:

df_head_tail = pd.DataFrame(columns=["head","relation","tail"])

for key,value in dict_keywords_per_product.items():
    parent_asin = key
    for line in value.split("\n"):
        match = re.search(r'\("([^"]+)"\s*,\s*"([^"]+)"\)', line) 
        if match:
            parent, child = match.groups()
            df_head_tail = pd.concat([df_head_tail, pd.DataFrame({"head": [parent], "relation": ["is parent of"], "tail": [child]})])
    df_head_tail = pd.concat([df_head_tail, pd.DataFrame({"head": [parent_asin], "relation": ["is child of"], "tail": [parent]})])
df_head_tail


In [ ]:
import networkx as nx

G = nx.from_pandas_edgelist(df_head_tail, source='head', target='tail', edge_attr='relation')

nx.draw(G, with_labels=True)


Note use without mmr, to get more keywords regarding product ingredients 

mention in implementation with examples 

BERTopic    

Sideote:
- BERTopic 
    + use sentence transformer for embedding 
    + use HDBSCAN for hierarchical clustering 
    + use KeyBERTInspired for representation model, to make topics more descriptive
    + use centroid linkage for hierarchical clustering of topics


Mention in thesis 
- use KeyBERTInspired for representation model, to make topics more descriptive
- use centroid linkage for hierarchical clustering of topics
- Embedding models used:
    + all-MiniLM-L6-v2
        + standard lm embedding model, used in berttopic repo
    + intfloat/multilingual-e5-large-instruct
        + multilingual lm embedding model, open source, 2nd highest score on mteb for clustering for open source models with memory footpringt below 20 gb
    + Alibaba-NLP/gte-Qwen2-1.5B-instruct
        + sota open source embedding model, highest score on mteb for clustering for open source models with memory footpringt below 20 gb


In [2]:
from scipy.cluster import hierarchy as sch
from bertopic import BERTopic
import numpy as np 
from sklearn.cluster import KMeans
from hdbscan import HDBSCAN

random_seed = np.random.seed(42)

cluster_model = KMeans(n_clusters=5,random_state=random_seed)
topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2", hdbscan_model=cluster_model)
topics, probs = topic_model.fit_transform(metadata_filtered_df.source_text)

# Hierarchical topics
linkage_function = lambda x: sch.linkage(x, 'centroid', optimal_ordering=True)
mini_hierarchical_topics = topic_model.hierarchical_topics(metadata_filtered_df.source_text, linkage_function=linkage_function)

topic_model.visualize_hierarchy(hierarchical_topics=mini_hierarchical_topics)


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 4/4 [00:00<00:00, 487.70it/s]


In [3]:
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('intfloat/multilingual-e5-large-instruct')

# Create your representation model
representation_model = KeyBERTInspired()
cluster_model = KMeans(n_clusters=5)
# Use the representation model in BERTopic on top of the default pipeline
topic_model = BERTopic(embedding_model=embedding_model, representation_model=representation_model, hdbscan_model=cluster_model)
#topic_model = BERTopic(embedding_model=model)
topics, probs = topic_model.fit_transform(metadata_filtered_df.source_text)

linkage_function = lambda x: sch.linkage(x, 'centroid', optimal_ordering=True)
e5_hierarchical_topics = topic_model.hierarchical_topics(metadata_filtered_df.source_text, linkage_function=linkage_function)

topic_model.visualize_hierarchy(hierarchical_topics=e5_hierarchical_topics)


100%|██████████| 4/4 [00:02<00:00,  1.38it/s]


In [ ]:
from bertopic import BERTopic
from scipy.cluster import hierarchy as sch
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
model = SentenceTransformer("Alibaba-NLP/gte-Qwen2-1.5B-instruct", trust_remote_code=True)


topic_model = BERTopic(embedding_model=model)

topics, probs = topic_model.fit_transform(metadata_filtered_df.source_text)

linkage_function = lambda x: sch.linkage(x, 'centroid', optimal_ordering=True)
qwen_hierarchical_topics = topic_model.hierarchical_topics(metadata_filtered_df.source_text, linkage_function=linkage_function)

print(qwen_hierarchical_topics)


In [ ]:
# viz 
topic_model.visualize_hierarchy(hierarchical_topics=qwen_hierarchical_topics)


In [5]:
def triples_from_hierarchy(hierarchical_topics_df):
    """
    Process hierarchical topics dataframe to create triples in the form of
    (head, relation, tail) where head is parent_name, relation is 'subcategory of',
    and tail is either child_left_name or child_right_name.
    
    Args:
        hierarchical_topics_df: DataFrame containing hierarchical topic information
        
    Returns:
        List of triples representing the hierarchy relationships
    """
    triples = []
    relation = "subcategory of"
    
    for _, row in hierarchical_topics_df.iterrows():
        parent_name = row['Parent_ID']
        child_left_name = row['Child_Left_ID']
        child_right_name = row['Child_Right_ID']
        
        # Create triple for left child
        triples.append((child_left_name, relation, parent_name))
        
        # Create triple for right child
        triples.append((child_right_name, relation, parent_name))
    # df to triples
    embedding_triples_df = pd.DataFrame(triples, columns=['head', 'relation', 'tail'])
    return embedding_triples_df

# Generate triples from the hierarchical topics
#qwen_triples_df = triples_from_hierarchy(qwen_hierarchical_topics)
e5_triples_df = triples_from_hierarchy(e5_hierarchical_topics)
mini_triples_df = triples_from_hierarchy(mini_hierarchical_topics)



compare reqex similarity to obtain if the embedding models differ in hierarchy clusters

Metric used:
- levenshtein distance

Cosine similarity not applicable, because the similarity is sensitive to the embedding model used in calculation. Since the embedding models are different, the cosine similarity will not be able to compare the hierarchy clusters.

In [7]:
e5_hierarchical_topics

,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance
3,8,gel_shampoo_moisturizing_ingredients_nail,"[0, 1, 2, 3, 4]",3,nails_nail_gel_manicure_acrylic,7,lotion_moisturizing_shampoo_ingredients_skin,0.424139
2,7,lotion_moisturizing_shampoo_ingredients_skin,"[0, 1, 2, 4]",2,headbands_headband_headwear_curly_accessories,6,lotion_shampoo_moisturizing_ingredients_products,0.376802
1,6,lotion_shampoo_moisturizing_ingredients_products,"[0, 1, 4]",5,lotion_moisturizing_skincare_skin_shampoo,4,lashes_mascara_eyelashes_eyeliner_lash,0.361107
0,5,lotion_moisturizing_skincare_skin_shampoo,"[0, 1]",1,moisturizer_moisturizing_hyaluronic_spf_skincare,0,lotion_shampoo_moisturizing_oils_conditioner,0.231790


In [12]:
mini_hierarchical_topics

,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance
3,8,price_store_and_for_description,"[0, 1, 2, 3, 4]",2,nail_gel_price_store_description,7,price_and_store_for_skin,0.408257
2,7,price_and_store_for_skin,"[0, 1, 3, 4]",6,price_store_and_for_skin,4,and_the_eyelashes_mascara_of,0.397488
1,6,price_store_and_for_skin,"[0, 1, 3]",1,hair_price_store_for_description,5,skin_and_price_store_for,0.346899
0,5,skin_and_price_store_for,"[0, 3]",0,skin_and_price_store_for,3,store_price_description_features_ounces,0.279788


In [11]:
import Levenshtein


def levenshtein_distance(df1, df2):

    def collect_unique_values(df):
        # collect from parent_name, child_left_name, child_right_name all unique values
        unique_values = []
        for column in ['Parent_Name', 'Child_Left_ID', 'Child_Right_ID']:
            unique_values.extend(df[column].unique())
        return unique_values
    
    unique_values_df1 = collect_unique_values(df1)
    unique_values_df2 = collect_unique_values(df2)

    return Levenshtein.ratio(unique_values_df1, unique_values_df2)

levenshtein_distance(e5_hierarchical_topics, mini_hierarchical_topics)


0.33333333333333337

In [22]:


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Fix the cosine_similarity function to handle matrix inputs correctly
def cosine_similarity_matrices(matrix1, matrix2):
    # Transpose the second matrix to align dimensions for dot product
    matrix2_transposed = matrix2.T
    # Calculate dot product between matrices
    dot_product = np.dot(matrix1, matrix2_transposed)
    
    # Calculate norms for each row in both matrices
    norm_matrix1 = np.linalg.norm(matrix1, axis=1, keepdims=True)
    norm_matrix2 = np.linalg.norm(matrix2, axis=1, keepdims=True)
    
    # Calculate cosine similarity
    cosine_sim = dot_product / (np.dot(norm_matrix1, norm_matrix2.T))
    
    return cosine_sim

# Update the calc_cosine_similarity function to use the correct similarity function
def calc_cosine_similarity_fixed(df1, df2, embedding_model):
    def collect_unique_values(df):
        # collect from parent_name, child_left_name, child_right_name all unique values
        unique_values = []
        for column in ['Parent_Name', 'Child_Left_Name', 'Child_Right_Name']:
            unique_values.extend(df[column].unique())
        return unique_values
    
    unique_values_df1 = collect_unique_values(df1)
    unique_values_df2 = collect_unique_values(df2)

    # embed unique values
    unique_values_df1_embed = embedding_model.encode(unique_values_df1)
    unique_values_df2_embed = embedding_model.encode(unique_values_df2)

    # Use the fixed cosine similarity function for matrices
    cosine_similarity_matrix = cosine_similarity_matrices(unique_values_df1_embed, unique_values_df2_embed).mean()

    return cosine_similarity_matrix

# Use the fixed function
calc_cosine_similarity_fixed(e5_hierarchical_topics, mini_hierarchical_topics, embedding_model)



0.36015797

Cosine similarity shows similar score to levenshtein distance

In [ ]:
 import numpy as np



# Example usage
vector1 = [1, 2, 3]
vector2 = [4, 5, 6]
similarity_score = cosine_similarity(vector1, vector2)
print(f"Cosine Similarity: {similarity_score}")


## 3.3 Classical hearst patterns

note:
illustrate with examples for part of speech  why not working properly
use entity output & graph